In [ ]:
from src.utils import get_data_env
from src.models import SpatialGNNModel
from src.dataloading import DeepDataLoader


/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
disease_name    = 'pertussis'
nuts_level      = 'nuts2'
min_date        = '2013-02-01'
max_date        = '2020-01-01'
split_trainval  = '2018-01-01'
split_valtest   = '2019-01-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 1
sequence_length = 1
lags            = 4

graphtype       = 'boolean_neighbors_nonself'

# training hparams
n_epochs        = 150
lr              = 0.0001
min_delta       = 0
loss            = 'quantile'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  : {'mode': 'min', 'factor': 0.8, 'patience': 5},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }


logged_zscore = DeepDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
logged_zscore.add_time_features()
logged_zscore.log_transform_target()
logged_zscore.set_splits(split_trainval, split_valtest)
logged_zscore.normalize()
logged_zscore.add_lagged_features(lags = lags)
logged_zscore.finalize()


Dataloader temporal windowing: extending data collection from 2013-02-01 to 2013-01-18 (+2 weeks)


EpiDataLoader(disease=pertussis, nuts_level=nuts2, min_date=2013-01-18, max_date=2020-01-01, horizon_size=1, horizon_leadtime=1, sequence_length=1)

In [ ]:
from src.configmanager.experimentconfigmanager import *

# training hparams
n_epochs        = 200
lr              = 0.000025
min_delta       = 0.0001
loss            = 'mse'


global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  : {'mode': 'min', 'factor': 0.6, 'patience': 5},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }

config = ExperimentConfig(
    name='experiment_comparing_gravity_models',
    data=DataConfig(
        disease_name        = 'pertussis',
        nuts_level          = 'nuts2',
        min_date            = '2013-02-01',
        max_date            = '2020-01-01',
        split_berlin        = False,

    ),
    timeseries=TimeSeriesConfig(
        sequence_length = 1,
        horizon_size    = 1,
        horizon_leadtime= 2,
        lags            = 4
    ),
    preprocessing=PreprocessingConfig(
        log_transform_target= True,
        add_time_features   = True,
        normalization_method='zscore'
    ),
    splits=SplitConfig(
        split_trainval  = '2018-01-01',
        split_valtest   = '2019-01-01'
    ),
    model= ['GATv2Model','SpatialGNNModel'],
    global_hparams = global_hparams,
    train_hparams  = {'verbose':2},
    graphs         = ['gravity1','gravity2','gravity3','gravity4','identity_graph','boolean_neighbors_self','boolean_neighbors_nonself'],
    baseline = True
)
runner = ExperimentRunner()
runner.define(config).execute('experiment_comparing_gravity_models')

Dataloader temporal windowing: extending data collection from 2013-02-01 to 2012-12-28 (+5 weeks)
Dataloader Snapshot: GraphDataLoaderEntry(x=(38, 3, 1), y=(38, 1), edge_index=(2, 114), edge_weight=(114,))

==    Training GATv2Model_gravity1    ==
Epoch 1 train loss: 0.9490, val loss: 1.0612 ✓ (new best)
Epoch 2 train loss: 0.8893, val loss: 1.0352 ✓ (new best)
Epoch 3 train loss: 0.8307, val loss: 0.9803 ✓ (new best)
Epoch 4 train loss: 0.8131, val loss: 0.9500 ✓ (new best)
Epoch 5 train loss: 0.7976, val loss: 0.9423 ✓ (new best)
Epoch 6 train loss: 0.7895, val loss: 0.9389 ✓ (new best)
Epoch 7 train loss: 0.7862, val loss: 0.9363 ✓ (new best)
Epoch 8 train loss: 0.7984, val loss: 0.9330 ✓ (new best)
Epoch 9 train loss: 0.7848, val loss: 0.9388 (patience: 1/20)


KeyboardInterrupt: 